# Trabajo Práctico - Diseño de Solución de Datos
## Sistema de Agricultura de Precisión basado en IoT y LoRaWAN

---

# 1. Análisis del caso de uso

## 1.1 Descripción del caso de uso
El proyecto consiste en el diseño de una solución de datos para un sistema de Agricultura de Precisión basado en dispositivos IoT y comunicaciones LoRaWAN. El sistema permite monitorear establecimientos agrícolas mediante dispositivos instalados en el campo, almacenar las mediciones generadas y brindar información para el monitoreo, la gestión del riego y futuras aplicaciones de Inteligencia Artificial.

## 1.2 Problema que busca resolver
Los dispositivos instalados en el campo generan información de forma continua y distribuida. La solución propuesta busca centralizar estos datos, administrar la infraestructura agrícola y conservar el historial de mediciones para facilitar el monitoreo, el análisis y la toma de decisiones.


## 1.3 Usuarios principales
El sistema contempla tres perfiles de usuario:
- **Operador:** consulta dispositivos, mediciones, gráficos y alarmas.
- **Configurador:** además de las funciones del Operador, administra configuraciones y alarmas.
- **Administrador:** administra usuarios, perfiles y permisos, además de todas las funciones anteriores.


## 1.4 Procesos y funcionalidades
El sistema deberá permitir:
- Administrar establecimientos, lotes, sectores y pivotes.
- Registrar y administrar dispositivos.
- Gestionar el historial de instalación de los dispositivos.
- Almacenar mediciones y datos de comunicación.
- Monitorear variables en tiempo real.
- Consultar información histórica.
- Administrar alarmas.
- Gestionar usuarios y permisos.


## 1.5 Información que gestiona el sistema
La solución administra información correspondiente a:
- Establecimientos agrícolas.
- Lotes, sectores y pivotes.
- Dispositivos IoT.
- Historial de instalaciones.
- Mediciones.
- Gateways y datos de comunicación.
- Usuarios y perfiles.
- Configuraciones y alarmas.

## 1.6 Riesgos relacionados con los datos
Los principales riesgos considerados son:
- Pérdida de mediciones.
- Inconsistencias en la ubicación de los dispositivos.
- Accesos no autorizados.
- Modificaciones indebidas de configuraciones.
- Crecimiento del volumen de datos.
- Pérdida de integridad de la información.


## 1.7 Principales decisiones de diseño
El diseño de la solución priorizará la integridad, trazabilidad y escalabilidad de los datos. Para ello, se considerará inicialmente utilizar una base de datos relacional PostgreSQL como plataforma principal, complementada con la extensión TimescaleDB para el almacenamiento eficiente de series temporales. Además, las variables medidas por los dispositivos se almacenarán en formato JSONB, permitiendo representar distintos tipos de mediciones de manera flexible.

---

# 2. Relevamiento de datos necesarios
La solución a desarrollar deberá almacenar o consultar los siguientes datos, clasificados según las categorías propuestas:

## 2.1 Datos estructurados
Se identifican los siguientes:
- Campos.
- Lotes.
- Sectores.
- Pivotes.
- Categorías de dispositivos.
- Tipos de dispositivos.
- Dispositivos.
- Instalaciones.
- Gateways.
- Usuarios.
- Perfiles.
- Alarmas.
- Configuraciones.

## 2.2 Datos semiestructurados
Las “mediciones”, ya que las variables medidas difieren según el tipo de dispositivo. Por ejemplo:
1. Estaciones meteorológicas:
    - temperatura.
    - humedad relativa.
    - velocidad y dirección del viento.
    - radiación solar.
    - precipitaciones.
2. sondas de suelo: por cada nivel (6 niveles):
    - humedad.
    - temperatura.
    - conductividad eléctrica.

Las variables medidas se almacenarán en formato JSONB, permitiendo registrar distintos conjuntos de valores sin modificar la estructura de la tabla de mediciones.


## 2.3 Datos no estructurados
No están contemplados por ahora.

## 2.4 Datos operacionales
Consideramos como datos operacionales a aquellos utilizados por la aplicación durante su funcionamiento diario, necesarios para realizar las tareas de monitoreo y administración.
Entre ellos se encuentran:
- Ubicación de sensores.
- Mediciones.
- Alarmas.
- Configuraciones.

## 2.5 Datos analíticos
En esta categoría básicamente se encuentra el histórico de mediciones, utilizado para realizar análisis y apoyar la toma de decisiones.


## 2.6 Datos sensibles
El sistema administra información que requiere protección, entre ella:
- Credenciales de acceso.
- Datos personales de los usuarios.
- Configuraciones del sistema.

## 2.7 Datos de auditoría y trazabilidad
El sistema conservará información que permita reconstruir eventos y realizar auditorías, incluyendo:
- Historial de instalación de los dispositivos.
- Historial de asignación de pivotes a lotes.

(Estos datos permitirán relacionar, por ejemplo, la influencia del riego en las variables medidas en suelo).

- Registros de recepción de mediciones.
- Fecha y hora de cada medición.

(Para corroborar que no se hayan perdido mensajes).
- Cambios realizados sobre configuraciones y alarmas.


## 2.8 Ejemplos de datos

---

# 4. Modelo conceptual

El modelo conceptual fija las entidades del dominio, sus atributos, las relaciones entre ellas, sus cardinalidades y las restricciones que deben cumplirse, con independencia de la tecnología con la que se implemente la solución.

Se construyó a partir del relevamiento del dominio real documentado en `docs/DetallesParaModelado.ipynb`, que recoge cómo opera efectivamente un establecimiento de agricultura de precisión: cómo se subdivide la superficie, cómo se instalan y reubican los dispositivos, cómo se transmiten las mediciones por LoRaWAN y cómo se definen las alarmas.

## 4.1 Alcance del modelo

El modelo conceptual describe **qué** información existe en el dominio y cómo se vincula, no **cómo** se almacena. En consecuencia, no incluye claves foráneas, tipos de datos del motor, índices ni tablas intermedias: todo eso corresponde al modelo de implementación (sección 5) y al modelo físico (sección 8).

Esa decisión tiene una consecuencia visible que conviene anticipar: el modelo conceptual tiene **16 entidades**, mientras que el esquema físico tiene **18 tablas**. La diferencia son las dos tablas asociativas que resuelven las relaciones muchos a muchos, que en este nivel se representan como relaciones y no como entidades (ver sección 4.6).

Sí se conserva la nomenclatura `snake_case` del esquema físico para nombrar entidades y atributos. Es una elección deliberada: aunque un modelo conceptual admitiría nombres de negocio en prosa, mantener un vocabulario único a lo largo del conceptual, el lógico, el físico y el DDL permite leer los cuatro artefactos en paralelo sin traducir nombres en cada salto.

El diagrama se mantiene como fuente Mermaid versionada en `docs/modelo_conceptual.mmd` y se exporta a `docs/modelo_conceptual.png`. Versionar el fuente además de la imagen permite revisar los cambios del modelo en el control de versiones, en lugar de comparar imágenes.

## 4.2 Diagrama entidad-relación

![Modelo conceptual del dominio](modelo_conceptual.png)

El diagrama usa notación de pata de gallo (*crow's foot*): el símbolo del extremo de cada línea indica la cardinalidad de ese lado. Una barra doble representa "exactamente uno", un círculo seguido de pata de gallo representa "cero o muchos". Las relaciones muchos a muchos aparecen con pata de gallo en ambos extremos.

## 4.3 Entidades principales

El dominio se organiza en cinco bloques temáticos, que son los mismos que estructuran el esquema físico (sección 8.1).

**Organización del establecimiento**

| Entidad | Descripción | Atributos relevantes |
| --- | --- | --- |
| `campo` | Establecimiento agrícola | `nombre`, `ubicacion`, `superficie` |
| `lote` | Parcela dentro de un campo | `nombre`, `superficie` |
| `sector` | Subdivisión de un lote | `nombre` |
| `pivote` | Máquina de riego por pivote central | `nombre`, `fabricante`, `modelo` |
| `asignacion_pivote` | Historial de qué pivote riega qué lote | `fecha_inicio`, `fecha_fin` |

**Dispositivos IoT**

| Entidad | Descripción | Atributos relevantes |
| --- | --- | --- |
| `categoria_dispositivo` | Agrupamiento mayor (suelo, riego, meteorológico) | `nombre` |
| `tipo_dispositivo` | Clase de equipo dentro de una categoría | `nombre` |
| `variable` | Magnitud física que un tipo de dispositivo sensa | `nombre`, `unidad` |
| `dispositivo` | Equipo físico instalado en el campo | `numero_serie`, credenciales LoRaWAN (`dev_eui`, `app_eui`, `app_key`), `intervalo_transmision`, `estado_operativo`, `estado_comunicacion` |
| `instalacion_dispositivo` | Historial de dónde estuvo instalado cada dispositivo | `fecha_inicio`, `fecha_fin` |

**Mediciones**

| Entidad | Descripción | Atributos relevantes |
| --- | --- | --- |
| `gateway` | Receptor LoRaWAN que retransmite las mediciones | `nombre` |
| `medicion` | Lectura individual enviada por un dispositivo | `fecha_hora`, `valores_medidos`, `rssi`, `snr`, `contador_mensajes` |

**Alarmas**

| Entidad | Descripción | Atributos relevantes |
| --- | --- | --- |
| `regla_alarma` | Condición que define cuándo generar una alarma | `descripcion`, `umbral_inferior`, `umbral_superior`, `habilitada` |
| `evento_alarma` | Ocurrencia concreta en que una regla se cumplió | `fecha_hora`, `valor_detectado` |

**Usuarios**

| Entidad | Descripción | Atributos relevantes |
| --- | --- | --- |
| `perfil` | Perfil de acceso (Operador, Configurador, Administrador) | `nombre`, `descripcion` |
| `usuario` | Persona que opera el sistema | `nombre`, `apellido`, `email`, `contrasena` |

Los atributos `rssi`, `snr` y `contador_mensajes` de `medicion` no describen la magnitud sensada sino la calidad del enlace de radio y la continuidad de la numeración de mensajes. Se incorporan como atributos de primer orden porque el caso requiere detectar pérdida de transmisiones (sección 2.7), lo que exige poder consultarlos y agregarlos igual que cualquier otra medición.

## 4.4 Relaciones y cardinalidades

| Relación | Cardinalidad | Lectura |
| --- | --- | --- |
| `campo` — `lote` | 1 : N | Un campo posee varios lotes; cada lote pertenece a un solo campo |
| `campo` — `pivote` | 1 : N | Un campo posee varios pivotes; cada pivote pertenece a un solo campo |
| `lote` — `sector` | 1 : N | Un lote se divide en varios sectores |
| `pivote` — `asignacion_pivote` | 1 : N | Un pivote acumula varias asignaciones a lo largo del tiempo |
| `lote` — `asignacion_pivote` | 1 : N | Un lote acumula varias asignaciones a lo largo del tiempo |
| `categoria_dispositivo` — `tipo_dispositivo` | 1 : N | Una categoría agrupa varios tipos |
| `tipo_dispositivo` — `dispositivo` | 1 : N | Un tipo clasifica varios dispositivos |
| `tipo_dispositivo` — `variable` | N : M | Un tipo sensa varias variables y una variable es sensada por varios tipos |
| `dispositivo` — `instalacion_dispositivo` | 1 : N | Un dispositivo acumula varias instalaciones a lo largo del tiempo |
| `campo` — `instalacion_dispositivo` | 1 : N | Un campo aloja varios dispositivos instalados |
| `sector` — `instalacion_dispositivo` | 1 : N | Un sector aloja varios dispositivos instalados |
| `pivote` — `instalacion_dispositivo` | 1 : N | Un pivote aloja varios dispositivos instalados |
| `dispositivo` — `medicion` | 1 : N | Un dispositivo genera muchas mediciones |
| `gateway` — `medicion` | 1 : N | Un gateway recibe muchas mediciones |
| `variable` — `regla_alarma` | 1 : N | Una variable es evaluada por varias reglas; cada regla evalúa una sola variable |
| `regla_alarma` — `dispositivo` | N : M | Una regla se aplica a varios dispositivos y un dispositivo tiene varias reglas |
| `regla_alarma` — `evento_alarma` | 1 : N | Una regla genera muchos eventos a lo largo del tiempo |
| `medicion` — `evento_alarma` | 1 : 0..N | Una medición puede no generar ningún evento, o generar varios |
| `perfil` — `usuario` | 1 : N | Un perfil agrupa varios usuarios; cada usuario tiene un solo perfil |

Las tres relaciones que vinculan `instalacion_dispositivo` con `campo`, `sector` y `pivote` son excluyentes entre sí: cada instalación participa de exactamente una de ellas. La notación entidad-relación no permite expresar esa exclusividad, por lo que se enuncia como restricción del dominio en la sección siguiente.

## 4.5 Restricciones del dominio

Las siguientes reglas surgen del negocio y deben cumplirse con independencia de la tecnología. Se indica además con qué mecanismo quedan garantizadas en la implementación, lo que permite distinguir las que el motor verifica por sí mismo de las que hoy dependen de la aplicación.

**Organización del establecimiento**

| Restricción | Cómo se garantiza |
| --- | --- |
| Un lote pertenece a un único campo | Clave foránea obligatoria |
| Un sector pertenece a un único lote | Clave foránea obligatoria |
| Un pivote pertenece a un único campo | Clave foránea obligatoria |
| Un pivote riega un solo lote a la vez | Índice único parcial sobre las asignaciones activas |
| Un lote es regado por un solo pivote a la vez | Sin mecanismo declarativo (ver nota) |

**Dispositivos**

| Restricción | Cómo se garantiza |
| --- | --- |
| Un dispositivo pertenece a un único tipo | Clave foránea obligatoria |
| Un tipo de dispositivo pertenece a una única categoría | Clave foránea obligatoria |
| Un tipo de dispositivo sensa al menos una variable | Sin mecanismo declarativo |
| Un dispositivo tiene como máximo una instalación activa | Índice único parcial |
| Cada instalación se asocia a exactamente un campo, sector o pivote | `CHECK` de exclusividad |
| Un dispositivo sin instalación activa no puede estar encendido | Sin mecanismo declarativo |
| Un dispositivo apagado no puede estar conectado | Sin mecanismo declarativo |
| Un dispositivo conectado debe estar encendido y tener instalación activa | Sin mecanismo declarativo |

**Mediciones**

| Restricción | Cómo se garantiza |
| --- | --- |
| Cada medición pertenece a un único dispositivo | Clave foránea obligatoria |
| Cada medición es recibida por un único gateway | Clave foránea obligatoria |
| Una medición no admite campos nulos | Parcial: `rssi` y `snr` admiten nulos |

**Alarmas**

| Restricción | Cómo se garantiza |
| --- | --- |
| Una regla de alarma define al menos un umbral | `CHECK` sobre umbral inferior y superior |
| Una regla de alarma evalúa una única variable | Clave foránea obligatoria |
| La variable de la regla debe pertenecer al tipo de dispositivo sobre el que se aplica | Sin mecanismo declarativo |

**Usuarios**

| Restricción | Cómo se garantiza |
| --- | --- |
| Existen únicamente los perfiles Operador, Configurador y Administrador | `CHECK` sobre el nombre del perfil |
| Todo usuario tiene un perfil asignado | Clave foránea obligatoria |
| Ningún usuario tiene más de un perfil | Cardinalidad del modelo |
| Un usuario no admite datos nulos | Restricciones `NOT NULL` |

**Nota sobre las restricciones sin mecanismo declarativo.** Las reglas marcadas de ese modo comparten una característica: involucran más de una fila o más de una tabla, y por lo tanto no son expresables con una clave foránea ni con un `CHECK` de fila. La coherencia entre `estado_operativo`, `estado_comunicacion` y la existencia de una instalación activa, o la pertenencia de una variable al tipo de dispositivo de la regla, requieren disparadores (`TRIGGER`) o validación en la capa de aplicación. El caso del lote regado por un solo pivote a la vez es distinto y más simple: el índice único parcial existente restringe el pivote pero no el lote, de modo que alcanza con un segundo índice análogo sobre el lote para cubrirlo. Estas restricciones se enuncian acá por pertenecer al dominio; su implementación se retoma en las secciones 8 y 13.

## 4.6 Decisiones de modelado

**Las relaciones N:M se representan como tales, sin resolver.** El diagrama muestra `tipo_dispositivo — variable` y `regla_alarma — dispositivo` como relaciones muchos a muchos, sin introducir entidades intermedias. Resolverlas mediante tablas asociativas es una decisión del modelo lógico (sección 5), no del dominio: la afirmación "un tipo de dispositivo sensa varias variables y una variable es sensada por varios tipos" es cierta independientemente de cómo se almacene. Se adoptó N:M y no 1:N porque `batería` es sensada por todos los tipos de dispositivo, y modelarla como 1:N obligaría a repetir esa variable por cada tipo, con la consiguiente redundancia y las anomalías de actualización asociadas. Esta es la razón por la que el modelo conceptual tiene 16 entidades y el esquema físico 18 tablas.

**`instalacion_dispositivo` y `asignacion_pivote` son entidades, no relaciones.** Ambas podrían parecer simples vínculos entre dos entidades, pero tienen atributos propios (`fecha_inicio`, `fecha_fin`) que no pertenecen a ninguno de los extremos: la fecha en que un dispositivo fue instalado no es un atributo del dispositivo ni del sector, sino del hecho de la instalación. Son entidades asociativas, y modelarlas así es lo que permite conservar el historial en lugar de solo el estado actual — un requisito explícito del caso (secciones 1.4 y 2.7).

**La instalación polimórfica se representa como tres relaciones excluyentes.** Un dispositivo se instala en un campo, un sector o un pivote. En el diagrama esto aparece como tres relaciones opcionales desde `instalacion_dispositivo`, porque la notación entidad-relación no puede expresar la exclusividad entre ellas. La regla "exactamente una de las tres" es una restricción del dominio (sección 4.5) y se garantiza en la implementación mediante un `CHECK` (sección 8.2). Se documenta acá para que la lectura del diagrama no induzca a pensar que una instalación puede apuntar a los tres lugares a la vez.

**Los valores medidos no se modelan como una relación entre `medicion` y `variable`.** El relevamiento de dominio enuncia que "una medición tiene múltiples variables". No se incorporó como relación explícita porque las variables que una medición puede contener quedan determinadas por el camino `medicion → dispositivo → tipo_dispositivo → variable`, que ya está en el modelo: agregar un vínculo directo duplicaría esa información y sugeriría una entidad de detalle que el diseño deliberadamente no tiene. El contenido variable de cada medición se representa como el atributo `valores_medidos`, cuya justificación como JSONB se desarrolla en las secciones 6 y 8.2.

---

# 5. Modelo de implementación según la tecnología elegida

La solución se implementa sobre una base de datos relacional —PostgreSQL, con la extensión TimescaleDB para la serie temporal de mediciones—, por lo que el modelo de implementación es un **modelo lógico relacional**. La justificación de esa elección tecnológica, comparada con las alternativas NoSQL y vectoriales, se desarrolla en la sección 7.

Esta sección presenta el modelo lógico derivado del modelo conceptual de la sección 4: tablas, columnas, claves primarias y foráneas, restricciones de integridad y resolución de las relaciones muchos a muchos. Los aspectos que dependen del motor —el particionado de la hipertabla, los índices GIN sobre JSONB y los índices únicos parciales— pertenecen al modelo físico y se tratan en la sección 8.

## 5.1 Del modelo conceptual al modelo lógico

La transformación del modelo conceptual al relacional siguió reglas sistemáticas, aplicadas de manera uniforme:

| Construcción conceptual | Traducción relacional |
| --- | --- |
| Entidad | Tabla, con identificador propio como clave primaria |
| Atributo | Columna con su tipo de dato y obligatoriedad |
| Relación 1:N | Clave foránea en el lado "muchos" |
| Relación N:M | Tabla asociativa con clave primaria compuesta (sección 5.4) |
| Entidad asociativa | Tabla propia, conservando sus atributos y su identificador |
| Relaciones excluyentes | Claves foráneas opcionales más una restricción que admite exactamente una |

El resultado son 18 tablas: las 16 entidades del modelo conceptual más las 2 tablas asociativas que resuelven las relaciones muchos a muchos.

Un punto que conviene aclarar es por qué `instalacion_dispositivo` y `asignacion_pivote` se traducen como tablas propias y no se absorben en otra. Aunque vinculan dos entidades, tienen atributos que no pertenecen a ninguno de los extremos (`fecha_inicio`, `fecha_fin`) y su cardinalidad es 1:N respecto de ambos lados: un dispositivo acumula varias instalaciones a lo largo del tiempo, y un sector aloja varios dispositivos. Absorberlas obligaría a guardar solo la ubicación vigente y perder el historial, que es un requisito del caso (secciones 1.4 y 2.7).

## 5.2 Diagrama lógico relacional

![Modelo lógico relacional](modelo_logico.png)

Cada tabla lista sus columnas con el tipo de dato, y marca las claves primarias (`PK`), foráneas (`FK`) y de unicidad (`UK`). Las columnas anotadas como `nullable` son las tres claves foráneas excluyentes de `instalacion_dispositivo`; el resto de las claves foráneas es obligatorio.

Al igual que el modelo conceptual, el diagrama se mantiene como fuente Mermaid versionada en `docs/modelo_logico.mmd` y se exporta a `docs/modelo_logico.png`.

## 5.3 Tablas y claves

El modelo comprende 18 tablas, organizadas en los mismos cinco bloques del modelo conceptual.

| Tabla | Clave primaria | Claves foráneas |
| --- | --- | --- |
| `campo` | `id_campo` | — |
| `lote` | `id_lote` | `id_campo` |
| `sector` | `id_sector` | `id_lote` |
| `pivote` | `id_pivote` | `id_campo` |
| `asignacion_pivote` | `id_asignacion` | `id_pivote`, `id_lote` |
| `categoria_dispositivo` | `id_categoria` | — |
| `tipo_dispositivo` | `id_tipo` | `id_categoria` |
| `variable` | `id_variable` | — |
| `tipo_variable` | (`id_tipo_dispositivo`, `id_variable`) | ambas columnas |
| `dispositivo` | `id_dispositivo` | `id_tipo` |
| `instalacion_dispositivo` | `id_instalacion` | `id_dispositivo`, y `id_campo` / `id_sector` / `id_pivote` (opcionales y excluyentes) |
| `gateway` | `id_gateway` | — |
| `medicion` | (`id_medicion`, `fecha_hora`) | `id_dispositivo`, `id_gateway` |
| `regla_alarma` | `id_regla` | `id_variable` |
| `alarma_dispositivo` | (`id_regla_alarma`, `id_dispositivo`) | ambas columnas |
| `evento_alarma` | `id_evento_alarma` | `id_regla` |
| `perfil` | `id_perfil` | — |
| `usuario` | `id_usuario` | `id_perfil` |

**Sobre la clave primaria de `medicion`.** Es la única tabla cuya clave primaria no responde a una decisión del modelo lógico sino a un requisito de la tecnología: al particionarse por tiempo, la columna de particionado debe formar parte de toda clave primaria o índice único de la tabla. De ahí que la clave sea (`id_medicion`, `fecha_hora`) y no `id_medicion` a secas. Es un caso de restricción del modelo físico que se refleja hacia arriba, y se detalla en la sección 8.2.

**Uso de identificadores subrogados.** Todas las tablas usan identificadores numéricos generados por el motor en lugar de claves naturales. `dispositivo` es el caso donde la alternativa era más tentadora: `numero_serie` y `dev_eui` identifican unívocamente un equipo. Se optó por el subrogado porque son datos administrados por el fabricante y por la red LoRaWAN, fuera del control del sistema; si un equipo se reaprovisiona o se corrige una carga errónea, cambiar una clave natural obligaría a propagar el cambio a todas las mediciones asociadas.

## 5.4 Resolución de las relaciones muchos a muchos

Las dos relaciones N:M del modelo conceptual se resuelven con tablas asociativas de clave primaria compuesta:

| Relación conceptual | Tabla asociativa | Clave primaria |
| --- | --- | --- |
| `tipo_dispositivo` — `variable` | `tipo_variable` | (`id_tipo_dispositivo`, `id_variable`) |
| `regla_alarma` — `dispositivo` | `alarma_dispositivo` | (`id_regla_alarma`, `id_dispositivo`) |

**Por qué clave compuesta y no un identificador propio.** Ambas tablas expresan la existencia de un vínculo y nada más: no tienen atributos adicionales ni son referenciadas por terceras tablas. Con clave primaria compuesta, el par no puede repetirse — sería imposible declarar dos veces que un mismo tipo de dispositivo sensa la misma variable. Agregar un identificador subrogado obligaría a definir además una restricción de unicidad sobre las dos columnas para conseguir la misma garantía, sumando una columna que nadie usaría.

**Qué se gana con `tipo_variable`.** Es la tabla que permite que `variable` funcione como catálogo compartido. `batería`, que todos los tipos de dispositivo sensan, existe como una única fila referenciada desde cada tipo, en lugar de repetirse tantas veces como tipos haya. Sin esta tabla, el modelo caería en la redundancia descrita en la sección 4.6.

**Qué se gana con `alarma_dispositivo`.** Permite definir una regla una sola vez y aplicarla a un conjunto de dispositivos. Una regla de "batería baja" con su umbral se escribe una vez y se asocia a todos los equipos que corresponda; cambiar el umbral es modificar una fila, no recorrer el parque de dispositivos.

## 5.5 Restricciones de integridad

El modelo lógico traslada las reglas del dominio (sección 4.5) a restricciones verificables por el motor. Se agrupan en cuatro mecanismos:

**Integridad de entidad.** Toda tabla tiene clave primaria. Las tablas asociativas la forman por composición de sus dos claves foráneas; el resto usa un identificador propio. `usuario.email` lleva además una restricción de unicidad, porque es el identificador con el que las personas inician sesión y admitir duplicados haría ambiguo el acceso.

**Integridad referencial.** Todas las claves foráneas del modelo son obligatorias salvo las tres de `instalacion_dispositivo` hacia `campo`, `sector` y `pivote`, que son opcionales por construcción: exactamente una de ellas está presente en cada fila. Esa obligatoriedad generalizada es deliberada — un lote sin campo, una medición sin dispositivo o un usuario sin perfil son estados que el dominio no admite, y dejar la columna nullable habilitaría representarlos.

**Integridad de dominio.** Se restringen por `CHECK` los atributos con conjunto de valores acotado (`dispositivo.estado_operativo`, `dispositivo.estado_comunicacion`, `perfil.nombre`), la exclusividad de la instalación polimórfica y la exigencia de que una regla de alarma defina al menos un umbral.

**Reglas no expresables declarativamente.** Como se detalla en la sección 4.5, un subconjunto de las restricciones del dominio involucra varias filas o varias tablas y no puede resolverse con claves ni con `CHECK` de fila. Requieren disparadores o validación en la capa de aplicación, y se retoman en las secciones 8 y 13.

La formulación concreta de cada restricción en el DDL, junto con los índices que las respaldan, se desarrolla en la sección 8.2.

## 5.6 Cobertura de los patrones de consulta

El modelo se validó contra las necesidades de consulta enunciadas en la sección 1.4, verificando que cada una tenga un camino de acceso resuelto por la estructura.

| Necesidad | Camino de acceso |
| --- | --- |
| Última medición de un dispositivo | `medicion` filtrada por `id_dispositivo`, ordenada por `fecha_hora` |
| Histórico de una variable en un rango de fechas | `medicion` por `id_dispositivo` y rango de `fecha_hora`, extrayendo la clave correspondiente de `valores_medidos` |
| Inventario de dispositivos por ubicación | `instalacion_dispositivo` activa (`fecha_fin IS NULL`) unida a `campo`, `sector` o `pivote` |
| Estado operativo de la red de dispositivos | `dispositivo` por `estado_operativo` / `estado_comunicacion`, unida a su instalación activa |
| Calidad del enlace y pérdida de mensajes | `medicion` agregando `rssi`, `snr` y las discontinuidades de `contador_mensajes` por dispositivo |
| Alarmas registradas en un período | `evento_alarma` por `fecha_hora`, unida a `regla_alarma` y a la `variable` evaluada |
| Reglas de alarma aplicables a un dispositivo | `alarma_dispositivo` por `id_dispositivo`, unida a `regla_alarma` |
| Qué pivote regaba un lote en una fecha dada | `asignacion_pivote` por `id_lote`, filtrando el intervalo que contiene esa fecha |

**Una consecuencia del diseño que conviene explicitar.** Consultar mediciones por ubicación —"todas las lecturas del sector B"— no se resuelve con una clave foránea directa, porque `medicion` no guarda dónde estaba el dispositivo: guarda qué dispositivo la generó. La ubicación se obtiene navegando a `instalacion_dispositivo` y seleccionando la instalación vigente en el momento de la medición, lo que constituye una unión temporal (la fecha de la medición debe caer dentro del intervalo `fecha_inicio`–`fecha_fin`).

Es el precio de conservar el historial: si un dispositivo se traslada de sector, las mediciones antiguas siguen atribuidas al sector donde efectivamente se tomaron, en lugar de reasignarse retroactivamente al nuevo. La alternativa —copiar la ubicación dentro de cada medición— haría la consulta más directa a costa de duplicar el dato y de introducir el riesgo de inconsistencia. Ese compromiso se retoma en las secciones 6 y 14.

---

# 8. Implementación mínima realizada

## 8.1 Esquema físico
El esquema físico se implementó en PostgreSQL 16 con la extensión TimescaleDB, en `db/estructura/01_create_tables.sql`. El script define 15 tablas, organizadas en cinco bloques:

1. **Organización del establecimiento**: `campo`, `lote`, `sector`, `pivote`, `asignacion_pivote`.
2. **Dispositivos IoT**: `categoria_dispositivo`, `tipo_dispositivo`, `variable`, `tipo_variable`, `dispositivo`, `instalacion_dispositivo`.
3. **Mediciones**: `gateway`, `medicion`.
4. **Alarmas**: `regla_alarma`, `evento_alarma`, `alarma_dispositivo`.
5. **Usuarios**: `perfil`, `usuario`.

El script se ejecuta sobre una base recién creada; no usa `IF NOT EXISTS` porque se apoya en el mecanismo de inicialización de la imagen de Postgres (ver 8.3), que solo corre una vez, sobre un volumen vacío.

## 8.2 Decisiones de diseño reflejadas en el DDL

**Hipertabla TimescaleDB para `medicion`.** La tabla de mediciones se convierte en hipertabla mediante `create_hypertable('medicion', 'fecha_hora')`, particionando internamente por tiempo. Es la tabla con mayor volumen esperado (una fila por dispositivo y transmisión), por lo que necesita el patrón de escritura y purga que ofrece TimescaleDB en lugar de una tabla relacional simple.

**JSONB para `valores_medidos`.** Cada tipo de dispositivo mide un conjunto distinto de variables (una estación meteorológica no mide lo mismo que una sonda de suelo). En vez de modelar una columna por variable o una tabla EAV, `medicion.valores_medidos` es JSONB, y se indexa con GIN (`ix_medicion_valores_gin`) para soportar filtros por contenido (operador `@>`) sin escanear toda la tabla.

**Instalación polimórfica de dispositivos.** Un dispositivo se instala en un campo, un sector o un pivote, nunca en más de uno a la vez. Se modeló con tres columnas FK nullable (`id_campo`, `id_sector`, `id_pivote`) en `instalacion_dispositivo` más un `CHECK` que exige que exactamente una esté presente:

```sql
CHECK (
    (id_campo IS NOT NULL)::INTEGER
    + (id_sector IS NOT NULL)::INTEGER
    + (id_pivote IS NOT NULL)::INTEGER = 1
)
```

Se prefirió esto a una tabla `ubicacion` genérica con `tipo` + `id_referencia` porque mantiene las FK reales de PostgreSQL (integridad referencial verificada por el motor), a costa de tener tres columnas nullable en vez de dos.

**Historial temporal.** `instalacion_dispositivo` y `asignacion_pivote` registran `fecha_inicio`/`fecha_fin`, donde `fecha_fin IS NULL` marca el registro activo. Un índice único parcial (`WHERE fecha_fin IS NULL`) garantiza que un mismo dispositivo o pivote no tenga más de una fila activa a la vez, sin depender de que la aplicación lo respete:

```sql
CREATE UNIQUE INDEX ux_instalacion_dispositivo_activa
    ON instalacion_dispositivo(id_dispositivo)
    WHERE fecha_fin IS NULL;
```

**Restricciones de integridad adicionales.** `regla_alarma` exige al menos un umbral definido (`umbral_inferior IS NOT NULL OR umbral_superior IS NOT NULL`); `dispositivo` restringe sus estados a valores válidos (`estado_operativo IN ('on', 'off')`, etc.) mediante `CHECK`; `usuario.email` es `UNIQUE`; las relaciones N:M (`tipo_variable`, `alarma_dispositivo`) se resuelven con tablas intermedias de clave primaria compuesta.

## 8.3 Entorno de ejecución

El entorno se levanta con `docker-compose.yml`, usando la imagen `timescale/timescaledb:latest-pg16`. El script `01_create_tables.sql` se monta en `/docker-entrypoint-initdb.d/`, que Postgres ejecuta automáticamente la primera vez que arranca sobre un volumen vacío — no hace falta correr el DDL a mano. Las credenciales y el nombre de la base se parametrizan por variables de entorno (`.env`, a partir de `.env.example`) para no versionar contraseñas.

Con esto, levantar el esquema completo desde cero se reduce a `docker compose up -d`.

---

# 9. Datos de ejemplo utilizados

## 9.1 Generación de datos sintéticos

Los datos de ejemplo se generan con un script en Python (`db/datos/generar_datos.py` + `db/datos/main.py`), usando `psycopg2` para insertar contra la base y `Faker` (locale `es_AR`) para los datos con apariencia realista (nombres, emails, ubicaciones). El script puebla las 15 tablas del esquema, respetando el orden de dependencias entre ellas y las restricciones definidas en el DDL.

## 9.2 Volumen generado

| Entidad | Cantidad |
| --- | --- |
| Campos | 3 |
| Lotes | 9 (3 por campo) |
| Sectores | 18 (2 por lote) |
| Pivotes | 6 (2 por campo) |
| Asignaciones de pivote | 6 |
| Categorías y tipos de dispositivo | según catálogo fijo (suelo, riego, meteorológico) |
| Dispositivos | 3 por tipo |
| Instalaciones de dispositivo | 1 por dispositivo (activa) |
| Gateways | 2 |
| Mediciones | 800 |
| Reglas de alarma | 4 |
| Eventos de alarma | 3 por regla (12 en total) |
| Usuarios | 6 |

Son volúmenes pequeños a propósito: alcanzan para validar entidades, relaciones y restricciones (la consigna no exige un dataset real ni de gran escala), sin complicar la verificación manual de los resultados.

## 9.3 Coherencia de los datos generados

Los datos no son aleatorios sin criterio: respetan la semántica del dominio.

- **`valores_medidos` varía según el tipo de dispositivo.** Una sonda de suelo genera JSON con humedad/temperatura/conductividad por nivel; una estación meteorológica genera temperatura, humedad relativa, viento, radiación y precipitación, coherente con lo relevado en la sección 2.2.
- **Las reglas de alarma tienen umbrales con sentido físico.** Por ejemplo, la regla de batería baja dispara con `umbral_inferior`, no `umbral_superior` (un valor de batería por debajo de un límite es el caso anómalo). Esto se verificó revisando los `valor_detectado` generados en `evento_alarma` contra el umbral de cada regla.
- **El historial temporal es consistente.** Cada dispositivo y cada pivote tiene exactamente una instalación/asignación activa (`fecha_fin IS NULL`) al finalizar la carga, reforzado por los índices únicos parciales del DDL.

## 9.4 Exportación de ejemplo

El script exporta una muestra de 100 mediciones (join contra `dispositivo` y `tipo_dispositivo`) a `data/ejemplos/mediciones.csv`, serializando `valores_medidos` como JSON válido (`json.dumps`) en lugar de la representación de string por defecto de Python, para que el CSV sea reutilizable por otras herramientas. 

# 10 Contexto de datos de prueba

Las consultas de esta sección se evaluaron sobre una carga sintética generada por el procedimiento `popular_tablas()`, con mediciones distribuidas en una ventana temporal de 1 año.  
La carga incluye múltiples dispositivos y gateways activos, con variabilidad por tipo de sensor y valores JSONB acordes al dominio (`humedad_suelo`, `temperatura_suelo`, `caudal`, `bateria`, etc.).  
Este enfoque permite validar consultas operativas y analíticas en un escenario más cercano a producción.

## 10.1 Humedad promedio por lote en los últimos 7 días

**Consulta:**
```sql
SELECT
    l.id_lote,
    l.nombre AS lote,
    ROUND(AVG((m.valores_medidos ->> 'humedad_suelo')::numeric), 2) AS humedad_promedio
FROM medicion m
JOIN dispositivo d
    ON d.id_dispositivo = m.id_dispositivo
JOIN instalacion_dispositivo i
    ON i.id_dispositivo = d.id_dispositivo
JOIN sector s
    ON s.id_sector = i.id_sector
JOIN lote l
    ON l.id_lote = s.id_lote
WHERE i.fecha_fin IS NULL
  AND m.fecha_hora >= NOW() - INTERVAL '7 days'
GROUP BY l.id_lote, l.nombre
ORDER BY humedad_promedio ASC;
```

Qué pregunta responde:
¿En qué lotes el nivel de humedad del suelo está más bajo en la última semana?

Por qué es útil:
Permite detectar zonas con riesgo de sequía o estrés hídrico, y facilita la toma de decisiones sobre riego y asignación de recursos. Es una consulta clave para la gestión operativa del campo.

## 10.2 Pivotes activos y su lote asignado

**Consulta:**
```sql
SELECT
    p.id_pivote,
    p.nombre AS pivote,
    l.id_lote,
    l.nombre AS lote,
    ap.fecha_inicio,
    ap.fecha_fin
FROM asignacion_pivote ap
JOIN pivote p
    ON p.id_pivote = ap.id_pivote
JOIN lote l
    ON l.id_lote = ap.id_lote
WHERE ap.fecha_fin IS NULL
ORDER BY p.id_pivote;
```

Qué pregunta responde:
¿Qué pivote está regando cada lote en este momento?

Por qué es útil:
Es relevante para coordinar la operación de riego y para verificar que la infraestructura activa esté correctamente asociada al lote correspondiente. También ayuda a detectar errores de configuración o cambios de asignación.

## 10.3 Dispositivos con batería baja

**Consulta:**
```sql
WITH ultimas_mediciones AS (
    SELECT
        m.id_dispositivo,
        m.valores_medidos,
        ROW_NUMBER() OVER (
            PARTITION BY m.id_dispositivo
            ORDER BY m.fecha_hora DESC
        ) AS rn
    FROM medicion m
)
SELECT
    d.id_dispositivo,
    td.nombre AS tipo_dispositivo,
    (um.valores_medidos ->> 'bateria')::numeric AS bateria_actual
FROM ultimas_mediciones um
JOIN dispositivo d
    ON d.id_dispositivo = um.id_dispositivo
JOIN tipo_dispositivo td
    ON td.id_tipo = d.id_tipo
WHERE um.rn = 1
  AND (um.valores_medidos ->> 'bateria')::numeric < 20
ORDER BY bateria_actual ASC;
```

Qué pregunta responde:
¿Qué dispositivos tienen batería crítica y requieren mantenimiento?

Por qué es útil:
Los sensores deben mantenerse operativos para asegurar continuidad en la medición. Esta consulta permite anticipar fallas y reducir el riesgo de pérdida de datos o interrupciones del monitoreo.

## 10.4 Eventos de alarma por regla

**Consulta:**
```sql
SELECT
    ra.id_regla,
    ra.descripcion,
    COUNT(*) AS total_eventos
FROM evento_alarma ea
JOIN regla_alarma ra
    ON ra.id_regla = ea.id_regla
WHERE ea.fecha_hora >= NOW() - INTERVAL '30 days'
GROUP BY ra.id_regla, ra.descripcion
ORDER BY total_eventos DESC;
```

Qué pregunta responde:
¿Cuáles son las alarmas más frecuentes en los últimos 30 días?

Por qué es útil:
Permite medir la criticidad del sistema y priorizar intervenciones. Es una consulta orientada a la gestión operativa, porque ayuda a identificar patrones de anomalías repetidas y posibles problemas estructurales.

## 10.5 Promedio de temperatura y humedad por sector

**Consulta:**
```sql
SELECT
    s.id_sector,
    s.nombre AS sector,
    l.id_lote,
    l.nombre AS lote,
    ROUND(AVG((m.valores_medidos ->> 'temperatura_suelo')::numeric), 2) AS temp_promedio,
    ROUND(AVG((m.valores_medidos ->> 'humedad_suelo')::numeric), 2) AS humedad_promedio
FROM medicion m
JOIN dispositivo d
    ON d.id_dispositivo = m.id_dispositivo
JOIN instalacion_dispositivo i
    ON i.id_dispositivo = d.id_dispositivo
JOIN sector s
    ON s.id_sector = i.id_sector
JOIN lote l
    ON l.id_lote = s.id_lote
WHERE i.fecha_fin IS NULL
  AND m.fecha_hora >= NOW() - INTERVAL '24 hours'
GROUP BY s.id_sector, s.nombre, l.id_lote, l.nombre
ORDER BY l.nombre, s.nombre;
```

Qué pregunta responde:
¿En qué sectores se registran condiciones más extremas de temperatura o humedad?

Por qué es útil:
Es clave para identificar zonas con comportamiento anómalo dentro del lote, optimizar el riego y detectar condiciones que puedan afectar el rendimiento del cultivo.

## 10.6 Valor del conjunto de consultas

El conjunto de consultas cubre los patrones operativos y analíticos más relevantes del caso de uso:

- **Agregación temporal y por entidad agrícola** (`GROUP BY` por lote/sector y filtros por ventana de tiempo de 24 h, 7 días y 30 días).
- **Trazabilidad de infraestructura activa** (estado vigente de pivotes e instalaciones con `fecha_fin IS NULL`).
- **Detección de condiciones críticas** (batería baja y frecuencia de alarmas).
- **Uso de datos semiestructurados** (`JSONB`) con extracción de variables desde `valores_medidos`.
- **Funciones de ventana** (`ROW_NUMBER`) para obtener la última medición por dispositivo.

En conjunto, estas consultas validan que la solución no solo almacena mediciones, sino que también soporta monitoreo operativo, mantenimiento preventivo y análisis para toma de decisiones.

## 10.7 Validación de rendimiento

Para validar desempeño se ejecutaron pruebas sobre la carga sintética generada por `popular_tablas()`, con mediciones distribuidas en una ventana de 1 año y múltiples dispositivos/gateways.

### 10.7.1 Metodología

1. Se ejecutó EXPLAIN ANALYZE sobre consultas representativas.
2. Se comparó el tiempo antes y después de crear índices adicionales de optimización en columnas de claves foráneas y filtros temporales.
3. Se evaluaron especialmente consultas con:
   - joins sobre tablas de alta cardinalidad,
   - filtros por fecha,
   - agregaciones.

### 10.7.2 Resultado resumido

| Consulta | Tiempo sin índices adicionales | Tiempo con índices | Mejora |
| --- | --- | --- | --- |
| 10.1 Humedad promedio por lote (7 días) | 22.798 ms | 2.360 ms | 89.65 % |
| 10.3 Dispositivos con batería baja | 44.520 ms | 38.566 ms | 13.37 % |
| 10.5 Temperatura/humedad por sector (24 h) | 5.147 ms | 0.999 ms | 80.59 % |

### 10.7.3 Conclusión de rendimiento

La estrategia de índices adicionales reduce el costo de joins y filtros temporales, y mejora la latencia de consultas operativas frecuentes.  
La mejora no es homogénea entre consultas: es muy alta en 10.1 y 10.5, y moderada en 10.3, donde el costo dominante está asociado al ordenamiento y al procesamiento de la ventana para obtener la última medición por dispositivo.  
En este contexto, el modelo resulta apto para monitoreo continuo y análisis sobre volúmenes altos de mediciones.